# Bacillota — árvores e evolução da esporulação

**Pergunta central:** como integramos as técnicas de laboratório e as ferramentas de bioinformática para investigar a diversidade das bactérias do solo?

Neste notebook você vai:

1. Comparar as árvores de 16S (com o isolado da turma), de 46 proteínas ribossomais e do GTDB.
2. Mapear nessas árvores os genes de esporulação e o fenótipo de esporulação descrito por Galperin et al. (2022).
3. Medir se a distribuição de cada gene acompanha a filogenia (D de Fritz e Purvis).
4. Reconstruir o estado ancestral da esporulação em Bacillota.
5. **Extra:** fazer o mesmo para o envelope celular (monodermes e didermes).

O Python cuida da interface e o **R** (`scripts/phylogeny.r`) faz os gráficos e as análises. Não é preciso rodar Bakta, OrthoFinder nem IQ-TREE aqui: as árvores de referência estão em `dados/`, e a árvore com o isolado vem do processo feito no notebook 01 (ou dos monitores).


## 1. Preparação

Arquivos necessários em `dados/`: `16S_final.tree.treefile`, `ribomarkers_final.tree.treefile`, `bac120_229.tree`, `tip_meta.tsv`, `heatmap_matrix2_spore_categories.tsv`, `gene_presence_absence_final_61.tsv` e `additional_gene_calls.tsv`. A árvore com o isolado, `16S_isolado.tree.treefile`, pode estar em `dados/` ou em `resultados/`.

As árvores 16S usam os identificadores `label`; as outras duas usam `assembly`. **Pontas sem metadados** (o isolado da turma e as referências que vocês adicionaram) aparecem na árvore com um **losango preto e o nome numa caixa amarela** e ficam de fora das estatísticas, porque não temos o genoma delas.

No heatmap, **amarelo dentro da matriz de genes** indica presença adicional inferida a partir de orthogroups (`additional_gene_calls.tsv`).

`Spore former` é a classificação fenotípica de Galperin (esporulante, não esporulante ou desconhecido); `expected` é o tipo de envelope segundo a literatura. Nenhum dos dois é previsto a partir dos genes.

**Enraizamento.** Todas as árvores são enraizadas da mesma forma: os genomas fora de Bacillota formam o outgroup. Se eles não formarem um grupo numa árvore, o R usa *midpoint* e avisa. A raiz usada aparece no título de cada figura e na coluna `rooting` das tabelas. Confira sempre, porque a raiz decide o que é "ancestral".


In [3]:
from pathlib import Path

def localizar_projeto():
    pasta = Path.cwd().resolve()
    for candidato in (pasta, *pasta.parents):
        if (candidato / "atividade").is_dir() and (candidato / "dados").is_dir():
            return candidato
    raise FileNotFoundError("Abra o JupyterLab na raiz do projeto ou em atividade/.")

PROJECT = localizar_projeto()
DATA_DIR = PROJECT / "dados"
RESULT_DIR = PROJECT / "resultados"
RESULT_DIR.mkdir(exist_ok=True)
print("Projeto:", PROJECT)

import subprocess
import shutil
import pandas as pd
from IPython.display import display, Image, Markdown
import ipywidgets as widgets

SCRIPT = PROJECT / 'scripts' / 'phylogeny.r'
GENES = DATA_DIR / 'gene_presence_absence_final_61.tsv'
ADDITIONAL = DATA_DIR / 'additional_gene_calls.tsv'
print('Script R:', SCRIPT.relative_to(PROJECT))
print('Rscript:', shutil.which('Rscript') or 'não encontrado')


Projeto: /home/marialuiza/Luiza/Colabs/bacillus/migração_git/claude/Bacillotav4
Script R: scripts/phylogeny.r
Rscript: /home/marialuiza/miniforge3/envs/bacillota-aula/bin/Rscript


In [4]:
TREE_FILES = {
    '16S': '16S_final.tree.treefile',
    'ribo': 'ribomarkers_final.tree.treefile',
    'gtdb': 'bac120_229.tree',
    'isolado': '16S_isolado.tree.treefile',
}
TREE_LABELS = {'16S': '16S rRNA (229 genomas)', 'ribo': '46 proteínas ribossomais',
               'gtdb': 'GTDB bac120', 'isolado': '16S + isolado da turma'}
MAIN_TREES = ['16S', 'ribo', 'gtdb']

def tree_path(kind):
    """Caminho da árvore em dados/ ou resultados/; None se não existir."""
    for folder in (DATA_DIR, RESULT_DIR):
        if (folder / TREE_FILES[kind]).is_file():
            return folder / TREE_FILES[kind]
    return None

necessary = [SCRIPT, GENES, ADDITIONAL, DATA_DIR / 'tip_meta.tsv',
             DATA_DIR / 'heatmap_matrix2_spore_categories.tsv',
             *(DATA_DIR / TREE_FILES[k] for k in MAIN_TREES)]
missing = [p for p in necessary if not p.is_file()]
if missing:
    raise FileNotFoundError('Arquivos necessários não encontrados:\n' + '\n'.join(map(str,missing)))
if not shutil.which('Rscript'):
    raise RuntimeError('Rscript não encontrado; ative o ambiente Conda/Mamba antes de abrir o JupyterLab.')
df = pd.read_csv(GENES, sep='\t')
assert df['assembly'].is_unique and len(df) == 229
assert len(df.columns) - 1 == 61
assert df.drop(columns='assembly').isin([0,1]).all().all()
print(f'{len(df)} genomas × {len(df.columns)-1} marcadores, sem valores indeterminados')
print('Árvore com o isolado:', tree_path('isolado') or 'ainda não disponível')


229 genomas × 61 marcadores, sem valores indeterminados
Árvore com o isolado: /home/marialuiza/Luiza/Colabs/bacillus/migração_git/claude/Bacillotav4/resultados/16S_isolado.tree.treefile


## 2. Explore as árvores

Selecione uma árvore e clique em **Desenhar árvore**. O R produz um PNG para visualização e um PDF para salvar. A opção **16S + isolado da turma** aparece quando o arquivo `16S_isolado.tree.treefile` estiver no lugar; clique em **Atualizar** depois de copiá-lo.

A prévia aparece numa região com rolagem. Para ler os nomes dos 61 genes, abra o PDF em `resultados/`.

Os pontos pretos nos nós indicam ramos bem sustentados. Nas árvores 16S e ribossomal, o critério é **UFBoot ≥ 95** (IQ-TREE); na GTDB, **bootstrap ≥ 85**. São medidas diferentes, por isso os limiares diferem.

O sombreado laranja marca **Bacillales**. Se a ordem não for monofilética numa árvore, cada clado formado só por Bacillales é sombreado separadamente, e o rótulo indica quantos clados são.

Se preferir explorar no iTOL, enraíze no mesmo outgroup (todos os genomas fora de Bacillota) para que as figuras sejam comparáveis.


In [5]:
tree_select = widgets.Dropdown(description='Árvore:', style={'description_width':'initial'},
                               layout=widgets.Layout(width='360px'))
refresh_button = widgets.Button(description='Atualizar', icon='refresh')
plot_button = widgets.Button(description='Desenhar árvore', icon='tree', button_style='primary')

def refresh_trees(_=None):
    available = [k for k in TREE_FILES if tree_path(k)]
    previous = tree_select.value
    tree_select.options = [(TREE_LABELS[k], k) for k in available]
    tree_select.value = previous if previous in available else ('isolado' if 'isolado' in available else 'gtdb')

refresh_trees()
refresh_button.on_click(refresh_trees)
# O heatmap é alto (229 linhas); uma área rolável evita que a prévia ocupe a página inteira.
plot_out = widgets.Output(layout=widgets.Layout(max_height='740px', overflow='auto', width='100%'))

def run_r(command, **options):
    """Chama scripts/phylogeny.r; cada opção vira --nome valor."""
    args = ['Rscript', str(SCRIPT), command, '--project', str(PROJECT)]
    for name, value in options.items():
        if value is not None:
            args += [f'--{name}', str(value)]
    return subprocess.run(args, cwd=PROJECT, capture_output=True, text=True, check=False)

def show_result(result):
    if result.stdout.strip(): print(result.stdout.strip())
    if result.stderr.strip(): print(result.stderr.strip()[-4000:])
    if result.returncode: print(f'Execução interrompida (código {result.returncode}).')

def show_image(name, width=1150):
    display(Image(filename=str(RESULT_DIR / f'{name}.png'), width=width))
    display(Markdown(f'**Arquivos completos:** [PNG](../resultados/{name}.png) · '
                     f'[PDF vetorial](../resultados/{name}.pdf).'))

def plot_clicked(_):
    with plot_out:
        plot_out.clear_output(wait=True)
        kind = tree_select.value
        if not tree_path(kind):
            print('Arquivo ausente:', TREE_FILES[kind]); return
        print('Construindo a árvore, aguarde...')
        res = run_r('plot', tree=kind)
        plot_out.clear_output(wait=True)
        show_result(res)
        if res.returncode == 0 and (RESULT_DIR / f'heatmap_{kind}.png').exists():
            show_image(f'heatmap_{kind}')
            print('Role a prévia verticalmente; para ler os 61 genes, abra o PDF.')

plot_button.on_click(plot_clicked)
display(widgets.VBox([widgets.HBox([tree_select, refresh_button, plot_button]), plot_out]))


## 3. Os genes de esporulação acompanham a filogenia? (D de Fritz)

O D compara o agrupamento de um caráter binário na árvore com duas referências: **D = 1** (distribuição aleatória) e **D = 0** (o esperado se o caráter evoluísse como uma variável contínua sob movimento browniano e fosse depois cortado em presente/ausente). D negativo indica agrupamento ainda mais forte que essa referência. `Pval1` e `Pval0` são testes **direcionais** contra cada referência.

Além dos 61 genes, a lista inclui o **fenótipo de esporulação (Galperin)**: compare o D do fenótipo com o de genes como `spo0A`.

**O que o D não diz.** O D mede se o caráter está agrupado na árvore, não *como* ele chegou lá. Um D baixo é compatível com herança vertical (com ou sem perdas). Um D perto de 1 pode vir de transferência horizontal, mas também de muitas perdas independentes, como a esporulação perdida em estafilococos, lactobacilos e *Listeria*. Para olhar a história do caráter, use a seção 4.

**Escopo.** Por padrão o D usa **só Bacillota**: os outgroups não têm genes de esporulação, e incluí-los mistura no D o contraste trivial "Bacillota vs. resto". Calcule também com os outgroups e compare.

Comece com **199 permutações**; use **999** para os valores finais. Valores `0.000` refletem a resolução das simulações, não probabilidades nulas. **Comparar as 3 árvores** usa só os genomas presentes nas três árvores de referência.


In [6]:
gene_names = [c for c in df.columns if c != 'assembly']
SPORE = [('Fenótipo de esporulação (Galperin)', 'spore')]
marker_select = widgets.Dropdown(options=SPORE + [(g, g) for g in gene_names],
    value='spo0A' if 'spo0A' in gene_names else 'spore',
    description='Marcador:', style={'description_width':'initial'}, layout=widgets.Layout(width='340px'))
permut_select = widgets.Dropdown(options=[('199 (rápido)', 199), ('999 (mais preciso)', 999)], value=199,
    description='Permutações:', style={'description_width':'initial'}, layout=widgets.Layout(width='240px'))
scope_select = widgets.Dropdown(options=[('Só Bacillota', 'bacillota'), ('Incluir outgroups', 'all')],
    value='bacillota', description='Genomas:', style={'description_width':'initial'},
    layout=widgets.Layout(width='260px'))
calc_button = widgets.Button(description='Calcular D', icon='play', button_style='success')
compare_button = widgets.Button(description='Comparar as 3 árvores', icon='exchange')
fritz_out = widgets.Output()

def fritz_clicked(compare=False):
    with fritz_out:
        fritz_out.clear_output(wait=True)
        kind, marker, scope = tree_select.value, marker_select.value, scope_select.value
        ks = MAIN_TREES if compare else [kind]
        absent = [TREE_FILES[k] for k in ks if not tree_path(k)]
        if absent:
            print('Árvores ausentes:'); print(*absent, sep='\n'); return
        print('Calculando Fritz D...')
        res = run_r('compare' if compare else 'fritz', tree=kind, marker=marker,
                    permut=permut_select.value, scope=scope)
        fritz_out.clear_output(wait=True)
        show_result(res)
        prefix = 'fritz_comparacao' if compare else f'fritz_{kind}'
        output = RESULT_DIR / f'{prefix}_{marker}_{scope}.tsv'
        if res.returncode == 0 and output.exists():
            result_table = pd.read_csv(output, sep='\t')
            display(result_table)
            if (result_table['status'] != 'OK').any():
                display(Markdown('**Atenção:** uma ou mais linhas falharam. Os valores `NaN` NÃO são estimativas de D; veja `status`.'))
            print('Salvo em:', output.relative_to(PROJECT))

calc_button.on_click(lambda _: fritz_clicked(False))
compare_button.on_click(lambda _: fritz_clicked(True))
print('A árvore usada é a selecionada na seção 2.')
display(widgets.VBox([widgets.HBox([marker_select, permut_select, scope_select]),
                      widgets.HBox([calc_button, compare_button]), fritz_out]))


A árvore usada é a selecionada na seção 2.


## 4. Reconstrução do estado ancestral da esporulação

Aqui a pergunta é direta: **o ancestral comum de Bacillota esporulava?**

O R ajusta um modelo de Markov para caracteres discretos (modelo Mk, `ape::ace`) na árvore escolhida na seção 2 e calcula a probabilidade de cada estado em cada nó interno. O caráter padrão é o **fenótipo de Galperin**; você também pode escolher um gene (presente/ausente).

Dois modelos são ajustados:

- **ER** (*equal rates*): ganhar e perder o caráter acontecem à mesma taxa.
- **ARD** (*all rates different*): as duas taxas são estimadas separadamente. Se uma delas sair ~0, o modelo está dizendo que a mudança só acontece num sentido (por exemplo, só perdas), e as probabilidades nos nós ficam perto de 0 ou 1.

A tabela traz uma linha por modelo: log-verossimilhança, AIC (menor é melhor; diferenças menores que ~2 não são evidência forte), as taxas estimadas e a probabilidade de cada estado no ancestral de Bacillota e na raiz. A figura usa o modelo de menor AIC.

**Cuidados.** A classificação de Galperin cobre só Firmicutes, então os outgroups ficam sem estado no fenótipo e a análise do fenótipo começa na base de Bacillota. Com um gene, os outgroups entram. Em qualquer caso, o resultado depende da raiz, da amostragem e do modelo.


In [ ]:
trait_select = widgets.Dropdown(options=SPORE + [('Envelope (expected)', 'expected')] + [(g, g) for g in gene_names],
    value='spore', description='Caráter:', style={'description_width':'initial'},
    layout=widgets.Layout(width='300px'))
model_select = widgets.Dropdown(options=[('ER e ARD (escolhe pelo AIC)', 'auto'), ('Só ER', 'ER'), ('Só ARD', 'ARD')],
    value='auto', description='Modelo:', style={'description_width':'initial'},
    layout=widgets.Layout(width='320px'))
anc_button = widgets.Button(description='Reconstruir', icon='history', button_style='warning')
anc_out = widgets.Output(layout=widgets.Layout(max_height='900px', overflow='auto', width='100%'))

def ancestral_clicked(_):
    with anc_out:
        anc_out.clear_output(wait=True)
        kind, trait = tree_select.value, trait_select.value
        if not tree_path(kind):
            print('Arquivo ausente:', TREE_FILES[kind]); return
        print('Ajustando os modelos, aguarde...')
        res = run_r('ancestral', tree=kind, trait=trait, model=model_select.value)
        anc_out.clear_output(wait=True)
        show_result(res)
        table = RESULT_DIR / f'ancestral_{kind}_{trait}.tsv'
        if res.returncode == 0 and table.exists():
            display(pd.read_csv(table, sep='\t'))
            show_image(f'ancestral_{kind}_{trait}', width=800)

anc_button.on_click(ancestral_clicked)
print('A árvore usada é a selecionada na seção 2.')
display(widgets.VBox([widgets.HBox([trait_select, model_select, anc_button]), anc_out]))


## 5. Extra: monodermes e didermes

Bacillota é conhecido como um filo de bactérias Gram-positivas, mas algumas linhagens têm **duas membranas** (são didermes), como as Gram-negativas clássicas. Os mesmos instrumentos das seções 3 e 4 servem para investigar isso:

- No heatmap, o bloco **Membrana externa/LPS** reúne 15 genes típicos do envelope diderme. Em quais classes eles aparecem juntos?
- Na seção 3, calcule o D de `lpxC` ou `bamA`.
- Na seção 4, escolha **Envelope (expected)** e veja o estado reconstruído para o ancestral de Bacillota. Os modelos ER e ARD concordam?

Um cuidado de vocabulário: **coloração de Gram e arquitetura do envelope não são a mesma coisa**. Há Bacillaceae que coram como Gram-negativas mas são monodermes (García-Miranda et al. 2026). Aqui falamos de **didermes**, ou seja, da presença de membrana externa.


## 6. Perguntas para discussão

**Árvores**

1. O isolado agrupou-se com quem? O que isso indica sobre sua identidade?
2. Há grupos com suporte baixo? O que isso significa?
3. Compare a posição de um mesmo clado nas árvores 16S, ribossomal e GTDB. Quais agrupamentos se mantêm? Por que as topologias diferem?

**Esporulação**

4. Existe um gene de esporulação presente em todos os formadores de endósporo da amostra? E em algum não esporulante?
5. Compare o D do fenótipo de esporulação com o de `spo0A` e o de um gene com distribuição mais irregular. O que os valores dizem, e o que eles **não** dizem?
6. A esporulação parece ter sido transmitida horizontalmente ou herdada com perdas? O que a reconstrução ancestral acrescenta ao D? Que análise a mais ajudaria?
7. Por que encontrar homólogos de genes de esporulação **não prova** que a bactéria esporula?
8. O que as presenças **amarelas** no heatmap acrescentam em relação às anotações originais?

**Extra: envelope**

9. Quais três classes de Bacillota são didermes? Você as manteria em Bacillota ou as realocaria? Por quê? Compare com Antunes et al. (2016, *eLife*) e Taib et al. (2020, *Nature Ecology & Evolution*).
10. Procure um gene "de membrana externa" presente em quase todos os genomas, inclusive monodermes. Ele serve para discriminar envelope? (Dica: `fabZ`.)